In [ ]:
!pip install gradio tensorflow opencv-python librosa mtcnn -q

import gradio as gr
import numpy as np
import cv2
import librosa
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.applications import Xception
from tensorflow.keras.applications.xception import preprocess_input as xception_preprocess
from tensorflow.keras.layers import (
    Input, Reshape, GlobalAveragePooling2D,
    Bidirectional, GRU, Dropout, Dense, BatchNormalization,
    Conv2D, MaxPooling2D
)
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2
from mtcnn import MTCNN

# ── Config ────────────────────────────────────────────────────
FRAME_COUNT       = 24
OUTPUT_FRAME_SIZE = (128, 128)
AUDIO_SAMPLE_RATE = 16000
AUDIO_DURATION    = 5
AUDIO_N_MELS      = 128

print("Loading MTCNN...")
detector = MTCNN()

# ── Rebuild exact same architectures from your Cell 6 ─────────
def build_video_model():
    video_input = Input(shape=(FRAME_COUNT, 128, 128, 3), name="video_input")

    base_cnn = Xception(weights='imagenet', include_top=False,
                        input_shape=(128, 128, 3), pooling='avg')
    for layer in base_cnn.layers[:-30]:
        layer.trainable = False

    def apply_cnn_to_frames(inputs):
        B = tf.shape(inputs)[0]
        frames = tf.reshape(inputs, (B * FRAME_COUNT, 128, 128, 3))
        frames = xception_preprocess(frames * 255.0)
        features = base_cnn(frames, training=False)
        feat_dim = features.shape[-1]
        return tf.reshape(features, (B, FRAME_COUNT, feat_dim))

    x = tf.keras.layers.Lambda(apply_cnn_to_frames,
                                name="xception_timedist")(video_input)
    x = BatchNormalization(name="bn_temporal")(x)
    x = Bidirectional(GRU(128, return_sequences=False,
                          kernel_regularizer=l2(0.005),
                          recurrent_regularizer=l2(0.001)),
                      name="bigru")(x)
    x = Dropout(0.5, name="drop_vid")(x)
    x = Dense(64, activation='relu', name="fc_vid")(x)
    x = Dropout(0.4, name="drop_vid2")(x)
    output = Dense(2, activation='softmax', name="out_vid")(x)
    return Model(inputs=video_input, outputs=output, name="video_model")

def build_audio_model():
    audio_input = Input(shape=(128, 128, 1), name="audio_input")
    x = Conv2D(32, (3,3), activation='relu', padding='same')(audio_input)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2,2))(x)
    x = Conv2D(64, (3,3), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2,2))(x)
    x = Conv2D(128, (3,3), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2,2))(x)
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)
    output = Dense(2, activation='softmax', name="out_aud")(x)
    return Model(inputs=audio_input, outputs=output, name="audio_model")

# ── Build then load weights only ──────────────────────────────
print("Building model architectures...")
video_model = build_video_model()
audio_model = build_audio_model()

print("Loading weights...")
video_model.load_weights("best_video_model.keras")
audio_model.load_weights("best_audio_model.keras")
print("✅ Both models ready!")

# ── Face extraction ───────────────────────────────────────────
def extract_face(frame):
    try:
        results = detector.detect_faces(frame)
        if results:
            x, y, w, h = results[0]['box']
            x, y = max(0, x), max(0, y)
            if w > 0 and h > 0:
                face = frame[y:y+h, x:x+w]
                if face.size > 0:
                    return cv2.resize(face, OUTPUT_FRAME_SIZE)
    except Exception:
        pass
    return cv2.resize(frame, OUTPUT_FRAME_SIZE)

# ── Video preprocessing ───────────────────────────────────────
def preprocess_video(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None
    frames = []
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step  = max(total // FRAME_COUNT, 1)
    for i in range(FRAME_COUNT):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i * step)
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(extract_face(frame))
    cap.release()
    if len(frames) == 0:
        return None
    while len(frames) < FRAME_COUNT:
        frames.append(frames[-1])
    frames = np.array(frames[:FRAME_COUNT], dtype=np.float32) / 255.0
    return np.expand_dims(frames, axis=0)   # (1, 24, 128, 128, 3)

# ── Audio preprocessing ───────────────────────────────────────
def preprocess_audio(audio_path):
    try:
        y, sr = librosa.load(audio_path, sr=AUDIO_SAMPLE_RATE,
                             duration=AUDIO_DURATION)
        mel    = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=AUDIO_N_MELS)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        mel_db = cv2.resize(mel_db.astype(np.float32),
                            (AUDIO_N_MELS, AUDIO_N_MELS))
        mel_min, mel_max = mel_db.min(), mel_db.max()
        mel_db = (mel_db - mel_min) / (mel_max - mel_min + 1e-8)
        return np.expand_dims(np.expand_dims(mel_db, -1), 0)   # (1,128,128,1)
    except Exception as e:
        print(f"Audio error: {e}")
        return None

# ── Prediction ────────────────────────────────────────────────
def predict_deepfake(video_file, audio_file):
    results  = []
    vid_prob = None
    aud_prob = None

    if video_file is not None:
        frames = preprocess_video(video_file)
        if frames is not None:
            pred     = video_model.predict(frames, verbose=0)[0]
            vid_prob = float(pred[1])
            bar_r = "█" * int(pred[0] * 20)
            bar_f = "█" * int(pred[1] * 20)
            results.append(
                f"### 🎬 Video Analysis\n"
                f"- **Real:** {pred[0]*100:.1f}%  `{bar_r}`\n"
                f"- **Fake:** {pred[1]*100:.1f}%  `{bar_f}`")
        else:
            results.append("### 🎬 Video Analysis\n❌ Could not process video.")

    if audio_file is not None:
        mel = preprocess_audio(audio_file)
        if mel is not None:
            pred     = audio_model.predict(mel, verbose=0)[0]
            aud_prob = float(pred[1])
            bar_r = "█" * int(pred[0] * 20)
            bar_f = "█" * int(pred[1] * 20)
            results.append(
                f"### 🔊 Audio Analysis\n"
                f"- **Real:** {pred[0]*100:.1f}%  `{bar_r}`\n"
                f"- **Fake:** {pred[1]*100:.1f}%  `{bar_f}`")
        else:
            results.append("### 🔊 Audio Analysis\n❌ Could not process audio.")

    if vid_prob is None and aud_prob is None:
        return "⚠️ Please upload at least a video or audio file."

    probs = [p for p in [vid_prob, aud_prob] if p is not None]
    fused = float(np.mean(probs))

    if len(probs) == 2:
        results.append(
            f"### 🔀 Combined Score\n"
            f"- **Real:** {(1-fused)*100:.1f}%\n"
            f"- **Fake:** {fused*100:.1f}%")

    verdict = f"## 🚨 VERDICT: FAKE\nConfidence: {fused*100:.1f}%" \
              if fused >= 0.5 else \
              f"## ✅ VERDICT: REAL\nConfidence: {(1-fused)*100:.1f}%"
    results.append(verdict)
    return "\n\n---\n\n".join(results)

# ── Gradio UI ─────────────────────────────────────────────────
with gr.Blocks(title="Deepfake Detector", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🕵️ Deepfake Detector
    Upload a **video** and/or **audio** file to detect if it's real or AI-generated.
    """)
    with gr.Row():
        with gr.Column():
            video_input = gr.Video(label="🎬 Upload Video")
            audio_input = gr.Audio(label="🔊 Upload Audio (optional)", type="filepath")
            detect_btn  = gr.Button("🔍 Analyse", variant="primary", size="lg")
        with gr.Column():
            output = gr.Markdown(value="*Results will appear here...*")

    detect_btn.click(fn=predict_deepfake,
                     inputs=[video_input, audio_input],
                     outputs=output)

demo.launch(share=True, debug=True)

Loading MTCNN...
Building model architectures...
83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Loading weights...
✅ Both models ready!


/tmp/ipykernel_1776/3254909730.py:193: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Deepfake Detector", theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://75504602e55a351ee5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
